# Quantitative Finance: Joshi Part 7

Implementations from *The Concepts and Practice of Mathematical Finance*
by Mark S. Joshi, using the `barter-joshi` crate (Rust) and its Python
bindings (`barter.joshi`).

## Topics Covered
- PayOff classes and dynamic dispatch
- Simple Monte Carlo pricing and convergence
- Anti-thetic variance reduction
- Black-Scholes closed-form pricing and all 5 Greeks
- CRR binomial trees (European and American)
- Newton-Raphson and bisection implied volatility solvers
- Path-dependent options: Asian, barrier, lookback

All examples use the Python bindings (`barter.joshi`).
The equivalent Rust implementations are available in the `barter-joshi` crate.



---
# Python Implementation


## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [1]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


Using barter from /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python/python


# Joshi Part 7: Payoffs & Monte Carlo (Python)

Python implementation of the PayOff classes and Simple Monte Carlo from
*The Concepts and Practice of Mathematical Finance* by Mark S. Joshi.

## Topics
- PayOff class hierarchy (Call, Put, Digital, DoubleDigital)
- Simple Monte Carlo pricing for European options
- Convergence table
- Anti-thetic variance reduction

In [2]:
import math
import numpy as np

In [3]:
from barter.joshi import Call, DigitalCall, Put
from barter.joshi import monte_carlo_antithetic, monte_carlo_convergence, simple_monte_carlo

## 1. PayOff Classes

Joshi uses C++ virtual inheritance for polymorphic payoffs.
In Python we use abstract base classes.

In [4]:
# Test payoffs
call = Call(100)
put = Put(100)
dc = DigitalCall(100)

for spot in [90, 100, 110]:
    print(f"S={spot}: Call={call(spot):.1f}, Put={put(spot):.1f}, DigitalCall={dc(spot):.1f}")

S=90: Call=0.0, Put=10.0, DigitalCall=0.0
S=100: Call=0.0, Put=0.0, DigitalCall=0.0
S=110: Call=10.0, Put=0.0, DigitalCall=1.0


## 2. Simple Monte Carlo

Simulate GBM paths: $S(T) = S(0) \exp\left((r - \frac{\sigma^2}{2})T + \sigma\sqrt{T}Z\right)$

where $Z \sim N(0,1)$.

In [5]:
# Price ATM call: S=100, K=100, r=5%, vol=20%, T=1
# call = Call(100)
price, se = simple_monte_carlo(call, 100, 0.05, 0.2, 1.0, 200_000)
print(f"MC Call Price: {price:.4f} ± {se:.4f}")
print(f"BS Exact:      10.4506")
print(f"95% CI:        [{price - 1.96*se:.4f}, {price + 1.96*se:.4f}]")

MC Call Price: 10.4634 ± 0.0331
BS Exact:      10.4506
95% CI:        [10.3985, 10.5283]


## 3. Convergence Table

Track the MC estimate at powers of 2 to observe convergence.

In [6]:
# call = Call(100)
table = monte_carlo_convergence(call, 100, 0.05, 0.2, 1.0, 2**18)

print(f"{'Paths':>10} {'Mean':>14} {'Std Error':>14}")
print("-" * 40)
for paths, mean, se in table:
    print(f"{paths:>10} {mean:>14.6f} {se:>14.6f}")

     Paths           Mean      Std Error
----------------------------------------
         2       4.528187       4.528187
         4      12.752656       5.174600
         8       7.055756       3.281749
        16       9.033944       2.493021
        32       9.725938       2.218365
        64       9.093417       1.405387
       128       8.147209       0.944616
       256       9.359880       0.912458
       512       9.918205       0.634136
      1024       9.854124       0.440045
      2048       9.743659       0.310619
      4096      10.160420       0.226746
      8192      10.348907       0.162882
     16384      10.352480       0.115669
     32768      10.596002       0.082290
     65536      10.444999       0.057792
    131072      10.426125       0.040899
    262144      10.454291       0.028875


## 4. Anti-thetic Variance Reduction

For each $Z$, also use $-Z$. The errors partially cancel.

In [7]:
# call = Call(100)

price_basic, se_basic = simple_monte_carlo(call, 100, 0.05, 0.2, 1.0, 100_000)
price_anti, se_anti = monte_carlo_antithetic(call, 100, 0.05, 0.2, 1.0, 100_000)

print(f"Basic MC:     {price_basic:.4f} ± {se_basic:.4f}")
print(f"Anti-thetic:  {price_anti:.4f} ± {se_anti:.4f}")
print(f"Variance reduction: {se_basic/se_anti:.1f}x")

Basic MC:     10.4205 ± 0.0468
Anti-thetic:  10.4673 ± 0.0331
Variance reduction: 1.4x


# Joshi Part 7: Black-Scholes & Greeks (Python)

Closed-form Black-Scholes pricing, all five Greeks, and put-call parity.

In [8]:
from barter.joshi import bs_call, bs_put, call_delta, call_rho, call_theta, gamma, vega

In [9]:
# ATM option: S=100, K=100, r=5%, vol=20%, T=1
S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

c = bs_call(S, K, r, sigma, T)
p = bs_put(S, K, r, sigma, T)
parity = c - p - (S - K * math.exp(-r * T))

print(f"Call price:  {c:.6f}")
print(f"Put price:   {p:.6f}")
print(f"Put-call parity residual: {parity:.2e}")

Call price:  10.450584
Put price:   5.573526
Put-call parity residual: 0.00e+00


## Greeks

In [10]:
print(f"{'Greek':<10} {'Value':>12}")
print("-" * 24)
print(f"{'Delta':<10} {call_delta(S, K, r, sigma, T):>12.6f}")
print(f"{'Gamma':<10} {gamma(S, K, r, sigma, T):>12.6f}")
print(f"{'Vega':<10} {vega(S, K, r, sigma, T):>12.6f}")
print(f"{'Theta':<10} {call_theta(S, K, r, sigma, T):>12.6f}")
print(f"{'Rho':<10} {call_rho(S, K, r, sigma, T):>12.6f}")

Greek             Value
------------------------
Delta          0.636831
Gamma          0.018762
Vega          37.524035
Theta         -6.414028
Rho           53.232482


## Spot vs Delta surface

In [11]:
spots = np.linspace(60, 140, 50)
deltas = [call_delta(s, 100, 0.05, 0.2, 1.0) for s in spots]
gammas = [gamma(s, 100, 0.05, 0.2, 1.0) for s in spots]

print("Spot → Delta and Gamma for K=100, vol=20%, T=1:")
for s, d, g in list(zip(spots, deltas, gammas))[::10]:
    print(f"  S={s:6.1f}  Δ={d:.4f}  Γ={g:.6f}")

Spot → Delta and Gamma for K=100, vol=20%, T=1:
  S=  60.0  Δ=0.0138  Γ=0.002929
  S=  76.3  Δ=0.1585  Γ=0.015839
  S=  92.7  Δ=0.4874  Γ=0.021518
  S= 109.0  Δ=0.7823  Γ=0.013503
  S= 125.3  Δ=0.9303  Γ=0.005341


# Joshi Part 7: Binomial Trees & Implied Volatility (Python)

Binomial tree pricing, American options, and implied volatility via Newton-Raphson.

In [12]:
from barter.joshi import binomial_american, binomial_european, bs_call, implied_vol_newton

## 1. CRR Binomial Tree

In [13]:
call_payoff = lambda s: max(s - 100, 0)
put_payoff = lambda s: max(100 - s, 0)

for steps in [50, 100, 200, 500]:
    c = binomial_european(call_payoff, 100, 0.05, 0.2, 1.0, steps)
    p = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
    print(f"Steps={steps:>4}: Call={c:.4f}, Put={p:.4f}")

print(f"\nBS exact: Call=10.4506, Put=5.5735")

Steps=  50: Call=10.4107, Put=5.5336
Steps= 100: Call=10.4306, Put=5.5536
Steps= 200: Call=10.4406, Put=5.5635
Steps= 500: Call=10.4466, Put=5.5695

BS exact: Call=10.4506, Put=5.5735


## 2. American Put Premium

In [14]:
steps = 500
euro_put = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
amer_put = binomial_american(put_payoff, 100, 0.05, 0.2, 1.0, steps)

print(f"European Put: {euro_put:.4f}")
print(f"American Put: {amer_put:.4f}")
print(f"Early exercise premium: {amer_put - euro_put:.4f}")

European Put: 5.5695
American Put: 6.0888
Early exercise premium: 0.5193


## 3. Newton-Raphson Implied Volatility

In [15]:
# Round-trip test: price at vol=25%, then recover
true_vol = 0.25
price = bs_call(100, 100, 0.05, true_vol, 1.0)
implied, iters = implied_vol_newton(price, 100, 100, 0.05, 1.0)
print(f"True vol:    {true_vol:.6f}")
print(f"Implied vol: {implied:.6f}")
print(f"Iterations:  {iters}")
print(f"Error:       {abs(implied - true_vol):.2e}")

True vol:    0.250000
Implied vol: 0.250000
Iterations:  4
Error:       1.67e-16


## 4. Implied Volatility Smile

In [16]:
# Generate a skewed vol smile
S, r, T = 100, 0.05, 1.0
strikes = np.linspace(80, 120, 9)

# Simulate market prices with a skew (lower strike → higher vol)
true_vols = 0.20 + 0.001 * (100 - strikes)  # simple skew
market_prices = [bs_call(S, K, r, v, T) for K, v in zip(strikes, true_vols)]

# Recover implied vols
print(f"{'Strike':>8} {'MktPrice':>10} {'TrueVol':>10} {'ImplVol':>10}")
print("-" * 42)
for K, mp, tv in zip(strikes, market_prices, true_vols):
    iv, _ = implied_vol_newton(mp, S, K, r, T)
    print(f"{K:>8.1f} {mp:>10.4f} {tv:>10.4f} {iv:>10.4f}")

  Strike   MktPrice    TrueVol    ImplVol
------------------------------------------
    80.0    24.8857     0.2200     0.2200
    85.0    20.7859     0.2150     0.2150
    90.0    16.9749     0.2100     0.2100
    95.0    13.5129     0.2050     0.2050
   100.0    10.4506     0.2000     0.2000
   105.0     7.8230     0.1950     0.1950
   110.0     5.6448     0.1900     0.1900
   115.0     3.9079     0.1850     0.1850
   120.0     2.5820     0.1800     0.1800


# Joshi Part 7: Path-Dependent Options (Python)

Asian options, barrier options, and lookback options via Monte Carlo.

In [17]:
from barter.joshi import asian_call_mc, barrier_up_out_call_mc, lookback_call_mc

## 1. Arithmetic Asian Call

In [18]:
price, se = asian_call_mc(100, 100, 0.05, 0.2, 1.0, 12, 50_000)
print(f"Arithmetic Asian Call: {price:.4f} ± {se:.4f}")
print(f"(Should be < European Call ≈ 10.45)")

Arithmetic Asian Call: 6.1559 ± 0.0382
(Should be < European Call ≈ 10.45)


## 2. Up-and-Out Barrier Call

In [19]:
for barrier in [120, 130, 150, 200]:
    price, se = barrier_up_out_call_mc(100, 100, barrier, 0.05, 0.2, 1.0, 252, 10_000)
    print(f"Barrier={barrier}: {price:.4f} ± {se:.4f}")

print(f"\nVanilla Call ≈ 10.45 (barrier → ∞)")

Barrier=120: 1.3612 ± 0.0345


Barrier=130: 3.5011 ± 0.0632


Barrier=150: 7.7696 ± 0.1099


Barrier=200: 10.4727 ± 0.1460

Vanilla Call ≈ 10.45 (barrier → ∞)


## 3. Lookback Call: payoff = S(T) - min S(t)

In [20]:
price, se = lookback_call_mc(100, 0.05, 0.2, 1.0, 252, 10_000)
print(f"Lookback Call: {price:.4f} ± {se:.4f}")
print(f"(Should be > Vanilla Call ≈ 10.45)")

Lookback Call: 16.7558 ± 0.1476
(Should be > Vanilla Call ≈ 10.45)


## Exotic Option Price Comparison

| Option | Price | Relation to Vanilla |
|--------|-------|---------|
| Vanilla Call | ≈ 10.45 | Baseline |
| Asian Call | < 10.45 | Averaging reduces volatility |
| Barrier (up-and-out) | < 10.45 | Knock-out reduces value |
| Lookback | > 10.45 | Optimal strike adds value |

---
## Rust Implementation

The Rust equivalents of all examples above are available in the `barter-joshi`
crate and can be run using the [evcxr Jupyter kernel](https://github.com/evcxr/evcxr).
See the `barter/examples/` directory and `barter-joshi/` crate for Rust source code.
